In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from src.CocoDataset import CocoDataset
from src.DetectionDataset import DetectionDataset, collate
from src.modelo import crearModelo

ds = CocoDataset("../data/raw/football-players/train")
loader = DataLoader(DetectionDataset(ds), batch_size=4, shuffle=True, collate_fn=collate)

modelo = crearModelo(ligero=True)
modelo.load_state_dict(torch.load("../outputs/modelo_mobilenet_8ep_b4.pt"))


optimizer = torch.optim.SGD(
    [p for p in modelo.parameters() if p.requires_grad],
    lr=0.005, momentum=0.9
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
modelo.train()

epocas = 8

for epoca in range(epocas):
    total = 0.0
    for n, (img, target) in enumerate(loader, 1):

        perdidas = modelo(list(img), list(target))
        loss = sum(perdidas.values())
        total += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"=== epoca {epoca}: coste medio {total/n:.4f} ===")



=== epoca 0: coste medio 0.7630 ===
=== epoca 1: coste medio 0.7650 ===
=== epoca 2: coste medio 0.7608 ===
=== epoca 3: coste medio 0.7571 ===
=== epoca 4: coste medio 0.7367 ===
=== epoca 5: coste medio 0.7283 ===
=== epoca 6: coste medio 0.7293 ===
=== epoca 7: coste medio 0.7257 ===


In [9]:
torch.save(modelo.state_dict(), "../outputs/modelo_16ep.pt")

In [10]:
from torchvision.transforms.functional import to_tensor
from src.metricas import evaluar

dsTest = CocoDataset("../data/raw/football-players/test")

modelo.eval()

PERSONAS = {2, 3, 4}
UMBRAL = 0.5

preds = {}

for i in sorted(dsTest.id2fichero):
    x = to_tensor(dsTest.imagen(i))

    with torch.no_grad():
        p = modelo([x])[0]

    cajas = [
        b for b, l, s in zip(p["boxes"].tolist(), p["labels"].tolist(), p["scores"].tolist())
        if int(l) in PERSONAS and s >= UMBRAL
    ]

    preds[i] = {"boxes" : cajas}

recall, precision = evaluar(dsTest, preds)
print(f"recall {recall:.3f}  precision {precision:.3f}")


recall 0.563  precision 0.509
